In [34]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

def scrapper_corinthians():
    # 1. Configuração do Navegador (undetected)
    options = uc.ChromeOptions()
    # Se o site ainda bloquear, comente a linha 'headless' para ver o que acontece
    # options.add_argument('--headless') 
    
    print("🚀 Iniciando navegador...")
    driver = uc.Chrome(options=options, version_main=146)
    
    url = "https://fbref.com/en/squads/bf4acd28/2025/Corinthians-Stats"
    
    try:
        print(f"🔍 Acedendo a: {url}")
        driver.get(url)
        
        # 2. Espera Humana: O FBRef deteta automação se for rápido demais
        # Esperamos entre 10 a 15 segundos para o Cloudflare validar a sessão
        time.sleep(random.randint(10, 15))
        
        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        
        # 3. Localizar a Tabela Standard Stats
        # O ID começa com 'stats_standard' (ex: stats_standard_24)
        table = soup.find("table", id=lambda x: x and x.startswith("stats_standard"))
        
        if not table:
            print("❌ Erro: Tabela não encontrada. O site pode ter bloqueado o acesso.")
            return
        
        data = []
        # Focamos no tbody para ignorar cabeçalhos de grupo
        rows = table.find("tbody").find_all("tr")
        
        print(f"📊 Processando {len(rows)} linhas encontradas...")
        
        for row in rows:
            # O nome do jogador é um cabeçalho de linha (th)
            player_node = row.find("th", {"data-stat": "player"})
            if not player_node:
                continue
                
            nome = player_node.get_text(strip=True)
            
            # Função para extrair dados das células td
            def get_stat(stat_name):
                cell = row.find("td", {"data-stat": stat_name})
                return cell.get_text(strip=True) if cell else "0"

            idade_raw = get_stat("age")
            posicao = get_stat("position")
            minutos = get_stat("minutes").replace(",", "")
            gols = get_stat("goals")
            assistencias = get_stat("assists")

            # Tratamento da Idade (FBRef usa Formato 'Anos-Dias', ex: 21-110)
            try:
                idade = int(idade_raw.split('-')[0]) if '-' in idade_raw else int(float(idade_raw or 0))
            except:
                idade = 0

            # 4. Filtro: Apenas jogadores Sub-21
            if 0 < idade <= 21:
                data.append({
                    "Jogador": nome,
                    "Idade": idade,
                    "Posição": posicao,
                    "Minutos": int(minutos or 0),
                    "Gols": int(gols or 0),
                    "Assistências": int(assistencias or 0)
                })

        # 5. Gerar DataFrame e Resultados
        df = pd.DataFrame(data)
        
        if not df.empty:
            print("\n✅ Sucesso! Jogadores Sub-21 encontrados:")
            print(df.sort_values(by="Minutos", ascending=False).to_string(index=False))
            
            # Opcional: Salvar em CSV
            # df.to_csv("corinthians_u21_stats.csv", index=False)
        else:
            print("\n⚠️ Tabela lida, mas nenhum jogador Sub-21 foi filtrado.")

    except Exception as e:
        print(f"❗ Ocorreu um erro: {e}")
        
    finally:
        print("\nFechando navegador...")
        driver.quit()

if __name__ == "__main__":
    scrapper_corinthians()

🚀 Iniciando navegador...


[03/27/26 17:26:10] INFO     patching driver executable                                              ]8;id=910064;file://c:\Users\victo\Downloads\teste_athletico_analista_dados\venv\Lib\site-packages\undetected_chromedriver\patcher.py\patcher.py]8;;\:]8;id=965895;file://c:\Users\victo\Downloads\teste_athletico_analista_dados\venv\Lib\site-packages\undetected_chromedriver\patcher.py#346\346]8;;\
                             C:\Users\victo\appdata\roaming\undetected_chromedriver\undetected_chrom               
                             edriver.exe                                                                           

🔍 Acedendo a: https://fbref.com/en/squads/bf4acd28/2025/Corinthians-Stats
📊 Processando 42 linhas encontradas...

✅ Sucesso! Jogadores Sub-21 encontrados:
         Jogador  Idade Posição  Minutos  Gols  Assistências
           Breno     19      MF     2522     1             1
      João Pedro     21      DF     1242     0             1
       Gui Negão     17   FW,MF      945     3             1
    Felipe Longo     19      GK      540     0             0
    Ryan Gustavo     21      MF      527     0             0
       Dieguinho     17   MF,FW      451     0             1
           Kayke     20   MF,FW      358     0             1
           André     18      MF      315     2             0
        Léo Mana     20      DF      225     0             0
            Kauê     20      GK       90     0             0
           Bahia     19      MF       49     0             0
Guilherme Amorim     17                0     0             0
 Gabriel Caipira     19      DF        0     0      